In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task3')
RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task3/data/IT.csv')
oot = pl.read_csv('task3/data/OOT.csv')
tx = pl.read_csv('task3/data/TRANSACTIONS.csv')

it = it.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
oot = oot.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
tx = tx.with_columns(pl.col('TRANSACTION_TIME').str.to_datetime().alias('TX_DT'))

customers = pl.concat([
    it.select('CUSTOMER_ID', 'TIME_DT'),
    oot.select('CUSTOMER_ID', 'TIME_DT')
])
print(f'IT: {it.shape}, OOT: {oot.shape}, TX: {tx.shape}')
print(f'Customers: {customers.shape[0]}')

In [ ]:
valid_tx = customers.join(tx, on='CUSTOMER_ID', how='left')
valid_tx = valid_tx.filter(pl.col('TX_DT') < pl.col('TIME_DT'))
valid_tx = valid_tx.with_columns(
    ((pl.col('TIME_DT') - pl.col('TX_DT')).dt.total_hours() / 24).alias('days_ago')
)
print(f'Valid TX after time filter: {valid_tx.shape[0]}')

In [ ]:
def agg_features(df, suffix=''):
    return df.group_by('CUSTOMER_ID', 'TIME_DT').agg([
        pl.len().alias(f'tx_count{suffix}'),
        pl.col('TRANSACTION_AMOUNT').mean().alias(f'amt_mean{suffix}'),
        pl.col('TRANSACTION_AMOUNT').std().alias(f'amt_std{suffix}'),
        pl.col('TRANSACTION_AMOUNT').max().alias(f'amt_max{suffix}'),
        pl.col('TRANSACTION_AMOUNT').min().alias(f'amt_min{suffix}'),
        pl.col('TRANSACTION_AMOUNT').sum().alias(f'amt_sum{suffix}'),
        pl.col('TRANSACTION_AMOUNT').median().alias(f'amt_median{suffix}'),
        pl.col('TRANSACTION_FEE').mean().alias(f'fee_mean{suffix}'),
        pl.col('TRANSACTION_FEE').std().alias(f'fee_std{suffix}'),
        pl.col('TRANSACTION_FEE').sum().alias(f'fee_sum{suffix}'),
        pl.col('days_ago').mean().alias(f'days_ago_mean{suffix}'),
        pl.col('days_ago').min().alias(f'days_ago_min{suffix}'),
        pl.col('days_ago').std().alias(f'days_ago_std{suffix}'),
        (pl.col('TRANSACTION_TYPE') == 'CREDIT').sum().alias(f'n_credit{suffix}'),
        (pl.col('TRANSACTION_TYPE') == 'DEBIT').sum().alias(f'n_debit{suffix}'),
        pl.col('TRANSACTION_CLASS').n_unique().alias(f'n_classes{suffix}'),
        pl.col('TRANSACTION_PLACE').n_unique().alias(f'n_places{suffix}'),
        pl.col('TRANSACTION_PURPOSE').n_unique().alias(f'n_purposes{suffix}'),
    ])

feat_all = agg_features(valid_tx, '')

windows = [30, 90, 180]
feat_windows = [feat_all]
for w in windows:
    sub = valid_tx.filter(pl.col('days_ago') <= w)
    feat_windows.append(agg_features(sub, f'_{w}d'))

features = feat_windows[0]
for fw in feat_windows[1:]:
    features = features.join(fw, on=['CUSTOMER_ID', 'TIME_DT'], how='left')

print(f'Features shape: {features.shape}')

In [ ]:
cat_cols = ['TRANSACTION_CLASS', 'TRANSACTION_TYPE', 'TRANSACTION_PURPOSE', 'TRANSACTION_PLACE']
cat_pivot_parts = []

for col in cat_cols:
    pivot = (
        valid_tx.group_by('CUSTOMER_ID', 'TIME_DT', col)
        .agg(pl.len().alias('cnt'))
        .pivot(on=col, index=['CUSTOMER_ID', 'TIME_DT'], values='cnt')
        .fill_null(0)
    )
    pivot = pivot.rename({c: f'{col}_{c}' for c in pivot.columns if c not in ['CUSTOMER_ID', 'TIME_DT']})
    cat_pivot_parts.append(pivot)

for cp in cat_pivot_parts:
    features = features.join(cp, on=['CUSTOMER_ID', 'TIME_DT'], how='left')

features = features.fill_null(0)
print(f'Features with categoricals: {features.shape}')

In [ ]:
ratios = features.with_columns([
    (pl.col('n_credit') / (pl.col('tx_count') + 1)).alias('credit_ratio'),
    (pl.col('amt_sum') / (pl.col('tx_count') + 1)).alias('avg_tx_size'),
    (pl.col('fee_sum') / (pl.col('amt_sum').abs() + 1)).alias('fee_to_amt_ratio'),
    (pl.col('tx_count_30d') / (pl.col('tx_count') + 1)).alias('recency_ratio_30'),
    (pl.col('tx_count_90d') / (pl.col('tx_count') + 1)).alias('recency_ratio_90'),
    (pl.col('amt_sum_30d') / (pl.col('amt_sum').abs() + 1)).alias('amt_recency_30'),
])

features = ratios
feat_cols = [c for c in features.columns if c not in ['CUSTOMER_ID', 'TIME_DT']]
print(f'Final feature count: {len(feat_cols)}')

In [ ]:
it_feat = it.join(features, on=['CUSTOMER_ID', 'TIME_DT'], how='left').fill_null(0)
oot_feat = oot.join(features, on=['CUSTOMER_ID', 'TIME_DT'], how='left').fill_null(0)

cutoff = it_feat.sort('TIME_DT')['TIME_DT'].quantile(0.8)
train = it_feat.filter(pl.col('TIME_DT') <= cutoff)
val = it_feat.filter(pl.col('TIME_DT') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')
print(f'Train target: {train["TARGET"].mean():.4f}, Val target: {val["TARGET"].mean():.4f}')

X_train = train.select(feat_cols).to_pandas()
y_train = train['TARGET'].to_numpy()
X_val = val.select(feat_cols).to_pandas()
y_val = val['TARGET'].to_numpy()
X_oot = oot_feat.select(feat_cols).to_pandas()

In [ ]:
lgb_tr = lgb.Dataset(X_train, y_train, free_raw_data=False)
lgb_va = lgb.Dataset(X_val, y_val, reference=lgb_tr, free_raw_data=False)

params = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'feature_pre_filter': False,
    'num_leaves': 31, 'learning_rate': 0.05,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'min_child_samples': 50,
}

model = lgb.train(params, lgb_tr, num_boost_round=1000, valid_sets=[lgb_va],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

p_lgb_val = model.predict(X_val)
val_gini_lgb = 2 * roc_auc_score(y_val, p_lgb_val) - 1
print(f'LGB: val_gini={val_gini_lgb:.4f}')

In [ ]:
imp = model.feature_importance(importance_type='gain')
imp_df = sorted(zip(feat_cols, imp), key=lambda x: -x[1])
for name, gain in imp_df[:20]:
    print(f'{name}: {gain:.1f}')

In [ ]:
val_gini = val_gini_lgb
p_oot_final = model.predict(X_oot)

preds = pl.DataFrame({'CUSTOMER_ID': oot_feat['CUSTOMER_ID'], 'SCORE': p_oot_final})
preds.write_csv('task3/predictions.csv')
print(f'OOT predictions: {preds.shape}')

with mlflow.start_run(run_name='v1_tx_agg_lgb'):
    mlflow.log_param('model', 'LightGBM')
    mlflow.log_param('n_features', len(feat_cols))
    mlflow.log_metric('val_gini', val_gini)
    mlflow.set_tag('task', 'task3')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')